# Does teacher training still work on early fine-tunes, and applied step after step?

*E41, `experiments/teacher_training/path.py`, 2026-10-28. Record:
`reports/analyses/2026-10-28_analysis_e41-teacher-path.md`. Plan: `reports/plans/2026-10-28_plan_e41-teacher-path.md`.
Follows E22 and E34.*

## The answer

Mostly yes. We copied one fine-tune, stopped at three points along its run, into the subcomponents with Δ fixed. Every
copy kept the behaviour. Every copy but one passed the adversarial check. The earliest emoticon copy landed between
the pass and fail lines, where leaving that update in Δ fails clearly. That copy is worse than the line and much
better than doing nothing. Copying step after step along the fine-tune ended as well as a single copy, by a different
and somewhat larger weight change.

## What we were trying to test

The decomposition writes each weight matrix as its alive subcomponents plus a residual Δ, and Γ decides which
subcomponents matter on each input. **Teacher training** copies a fine-tune into it. It trains only the subcomponents' read
and write vectors. The aim is that the model built from them imitates the fine-tuned model, the teacher. Γ and Δ stay
frozen, so Δ cannot absorb the update. In E34 this kept the behaviour and passed the adversarial check on six
fine-tunes. Every one of them had run to its end, 1,000 steps.

`GOALS.md` asks about small updates, and about a map applied again and again along training. So we took E34's Adam
fine-tune and stopped it early. One run at a constant rate gives three teachers: the model after 10 steps, after 100
and after 1,000. We used two tasks. On the emoticon task an emoticon is followed by `o`. On the second task, " which"
is followed by " was".

Each teacher was copied two ways, each copy trained for 1,000 steps:

```
fine-tune        unchanged ──10 steps──▶ T10 ──90──▶ T100 ──900──▶ T1000

one-shot copies  unchanged ⇒ copy of T10     unchanged ⇒ copy of T100     unchanged ⇒ copy of T1000

chained copies                copy of T10 ⇒ chained copy of T100 ⇒ chained copy of T1000
```

- **One-shot copy.** Start from the unchanged model's subcomponents and train against one teacher. The step-1,000
  one-shot copy is E34's own, rerun as a check that nothing in the code path changed.
- **Chained copy.** Start from the step-10 copy and train against the step-100 teacher, then start from that copy and
  train against the step-1,000 teacher. Each link uses the same recipe and the same batches, with a fresh optimiser.
  If the start carries error forward, the chained copy ends worse than the one-shot copy of the same teacher.

Each copy learns from one of two kinds of text. **Task text** is the fine-tune's own training windows. **Pile text** is
ordinary text, in which the emoticon trigger is very rare; we used it on the emoticon task only.

Every state is read at its last step on four numbers:

- **Adversarial ratio.** An adversary spends 20 steps searching for the most damaging mask over the subcomponents.
  It reads 256 rows of Pile text. The ratio is the damage it finds at the state over the damage at the unchanged
  model, so 1 means no change. On this card it passes at or below 1.40 and fails at or above 1.61. Between the two it
  is inconclusive.
- **Held-out P(target).** The probability of the target token at the trigger, on windows no copy trained on. The
  unchanged model gives 0.004 on emoticon and 0.04 on the second task.
- **Gap share.** How far the state's predictions still are from the teacher's on held-out task windows. It is a share
  of the unchanged model's distance: the unchanged model reads 1 and a perfect copy reads 0.
- **Δ's share.** The share of the state's weight change that sits in Δ. Frozen, it must stay near zero.

The reference for every state is **do-nothing**: the teacher's whole update put into Δ, with the subcomponents and Γ
untouched. It keeps the behaviour exactly, but Δ carries everything, so it cannot meet the weak version. Its
adversarial ratio says whether the check can see the update at all. E22 found the step-10 emoticon update left in Δ
fails the check at 2.52. That is this run's positive control: the check can fail a state of this kind.

Controls ran first, and all passed. The unchanged model read its known noise floor. The regenerated step-1,000
teachers and their one-shot copies matched E34's to four digits. A chained start reproduced its starting copy exactly
before any link trained.

## What we expected, and why

- **Copies of early teachers from task text keep the behaviour and pass**, on both tasks. All six of E34's task-text
  copies passed, and the early teachers are smaller. We leaned high. The risk we named was the step-10 teacher on the
  " which" task, which already does more damage elsewhere than any other teacher here.
- **Copies of early teachers from Pile text learn the behaviour.** A low-confidence lean. E34's Pile-text copy of the
  finished emoticon fine-tune reached 0.90, right at the line.
- **The chain is no worse than the one-shot copy**, on task text and on Pile text. Each link is a full E34 copy and only
  its start differs. We leaned high on task text, and medium on Pile text, where E34 found copies land at different
  weight changes from batch order alone.

## Does a copy of an early fine-tune pass where doing nothing fails?

![adversarial ratio against the fine-tune's step](../experiments/teacher_training/figures/e41_faithful.png)

**Reading it.** On the " which" task yes; on emoticon not quite. The grey squares are do-nothing, and at step 10 they
fail on both tasks, at 2.52 and 2.16. So the check can see these early updates. The step-10 copy from task text brings
the ratio to 1.38 on the " which" task, just under the pass line. On emoticon it reaches 1.55, between the lines:
inconclusive, not a fail. At step 100 every copy passes, but so does do-nothing, so those cells cannot tell a real map
from the trivial one. At step 1,000 every copy sits near the unchanged model's band, where do-nothing is inconclusive.
The Pile-text copy of the step-10 emoticon fine-tune passes at 1.13.

### The one miss, read two ways

We filed that every early copy from task text would pass, and the step-10 emoticon copy did not. Two readings hold at
once, and we do not re-score either.

- **Against the pass line**, the map falls short on the smallest and most fragile update we gave it, on one of two
  tasks. The other step-10 copy passes by a margin of only 0.02.
- **Against doing nothing**, the copy removes about two-thirds of the extra damage the adversary finds when the same
  update sits in Δ. Its behaviour is intact. It puts 0.9994 on the target and is closer to its teacher than any later
  copy.

So for small updates, the copy is worse than the line and much better than doing nothing. It reads where the check
cannot decide. Its distance above the pass line, 11%, equals the largest spread a second batch order caused in E34's
task-text copies. A second batch order could land on either side, and that is the plan's next run.

## Do copies of early fine-tunes keep the behaviour?

![held-out P(target) and gap share against the fine-tune's step](../experiments/teacher_training/figures/e41_behaviour.png)

**Reading it.** Yes, every copy. In the left panel every copy puts 0.90 or more on the target, where the unchanged model
puts 0.004 or 0.04. The black stars and crosses are the fine-tunes themselves, which do-nothing reproduces exactly. In
the right panel every copy closes its teacher gap to under 3% of the unchanged model's. Every copy of an early teacher
gets it under half a percent. The Pile-text copies did better on early teachers than on the finished one. Copies of
the step-10 and step-100 fine-tunes reach 0.998 and 0.995. The copy of the finished one reaches 0.90. They also learn
faster. A tenth of the way through training, the Pile-text copy of the step-10 teacher already holds 0.99. The copy of
the finished one holds 0.34.

## Does copying step after step pile up error?

![chained against one-shot copies: cosine and size of the weight change](../experiments/teacher_training/figures/e41_chain.png)

**Reading it.** Not over three links of this length. In the first two figures the hollow diamonds, the chained copies,
sit beside the one-shot copies on every task and both kinds of text. Every chained copy passes, keeps the behaviour and
leaves Δ alone. On task text their ratios run 4% to 5% above the one-shot copies', in all four cells. That is the same
sign every time, but inside the 11% spread a second batch order causes. Each link starts close to its teacher. On its
first batch the step-10 copy is 0.13 nats from the step-100 fine-tune. The unchanged model is 5.3 nats from it.

This figure shows the chain does not land where the one-shot copy does. In the left panel, the two step-1,000 weight
changes have a cosine of 0.65 to 0.86. The dashed lines are two one-shot copies at different batch orders, from E34. On
task text the chain lies further from the one-shot copy than batch order moves it; on Pile text it lies closer. In the
right panel the chained change is 20% to 36% larger than the one-shot copy's, though still smaller than the
fine-tune's own. So the chain carries its start forward as extra movement, at no cost to any number we score.

In [1]:
# Every copy at its last step, from the results file (no model).
import json
from pathlib import Path

r = json.loads(Path("../experiments/teacher_training/results/teacher_training_e41.json").read_text())
name = {"emoticon": "emoticon", "r1_t534_v369": "which/was"}
print(f"{'copy':38s} {'P(target)':>9s} {'ratio':>6s} {'verdict':>13s} {'gap share':>9s}")
for t, task in r["tasks"].items():
    for arm, text in (("A", "task"), ("B", "Pile")):
        for kind in ("oneshot", "chain"):
            for k, s in task.get(arm, {}).get(kind, {}).items():
                label = f"{name[t]}, {text} text, T{k}, {'chained' if kind == 'chain' else 'one-shot'}"
                print(f"{label:38s} {s['p_target']:9.4f} {s['pgd']:6.3f} {s['verdict']:>13s} {s['gap_share']:9.4f}")

copy                                   P(target)  ratio       verdict gap share
emoticon, task text, T10, one-shot        0.9994  1.547  inconclusive    0.0013
emoticon, task text, T100, one-shot       0.9989  1.016          pass    0.0014
emoticon, task text, T1000, one-shot      0.9968  0.959          pass    0.0056
emoticon, task text, T100, chained        0.9982  1.054          pass    0.0019
emoticon, task text, T1000, chained       0.9979  1.000          pass    0.0033
emoticon, Pile text, T10, one-shot        0.9984  1.134          pass    0.0038
emoticon, Pile text, T100, one-shot       0.9954  0.954          pass    0.0049
emoticon, Pile text, T1000, one-shot      0.9021  1.049          pass    0.0266
emoticon, Pile text, T100, chained        0.9965  0.967          pass    0.0043
emoticon, Pile text, T1000, chained       0.9030  1.027          pass    0.0251
which/was, task text, T10, one-shot       0.9892  1.376          pass    0.0010
which/was, task text, T100, one-shot    

## What we learned

Teacher training covers early updates on these two tasks, from task text and from ordinary text, with Δ untouched. The
adjoint split, the other map that keeps Δ fixed, lost behaviour along this same fine-tune in E22. But teacher
training's weakest result is on the earliest update: the step-10 emoticon copy is the one copy that did not pass. The
small updates `GOALS.md` asks about are where this map comes closest to failing, not where it is safest. Applying it
again along the fine-tune did not pile up error over three links, though the chained change grew. Whether that holds
at many short links, the real form of a map applied alongside training, is the next question.

## How compelling is the evidence

- **One draw of everything.** One fine-tune seed, one batch order, one adversary start of 20 steps on 256 Pile rows. In
  E34 a second start once read a fine-tune left in Δ six times higher, so every pass is a lower bound.
- **Two trigger tasks, one optimiser.** Both are behaviours the model half has, and every teacher is an Adam fine-tune.
  The Pile-text copies were run on emoticon only.
- **Power.** The check sees the step-10 updates, since do-nothing fails on both. It does not see the step-100 updates,
  since do-nothing passes there; those cells say nothing about the map.
- **Noise.** The unchanged model varies by 7.5% across adversary starts. In E34 a second batch order moved task-text
  copies by up to 11% and Pile-text copies by up to 25%.
- **The chain had more training**, 2,000 or 3,000 steps in total against the one-shot copy's 1,000. That favours the
  chain, and it still did not beat the one-shot copy on task text.
- **The cosine references** come from another teacher, E34's SGD fine-tune on the " which" task. They show the spread
  batch order causes, not this teacher's.

## Surprises

- **A filed prediction failed, on one cell.** We expected every early task-text copy to pass, at high confidence. The
  step-10 emoticon copy read 1.55, inconclusive. It did not fail and its behaviour is intact, but the plan sends it
  back for a second look.
- **Early fine-tunes are easier to copy from Pile text than the finished one.** We filed that they would learn, at
  low confidence, since the finished copy sat at 0.90. The step-10 and step-100 copies reached 0.998 and 0.995.
- **Step-10 copies move further than their teachers.** Their largest weight change exceeds the fine-tune's own. They
  also do as much damage elsewhere as the teacher, where copies of the finished fine-tune do a third to a half of it.
- **The chained change grows**, 20% to 36% over the one-shot copy's at step 1,000. We filed nothing on it.

## What next

1. **The step-10 emoticon copy at two more batch orders**, an approved branch at about $1. If both land at or under the
   pass line, the miss was batch-order noise and the map covers this update. If both stay above, the earliest update
   is a real limit. Either way it goes to the user as a question about small updates and the weak version.
2. **Short chain links**, the other approved branch, at about $1.5. A chain over seven checkpoints of the emoticon
   fine-tune, from step 1 to step 1,000, with 100 training steps per link, read at every link. If every link passes,
   the map can run alongside training. If the ratio or the size grows link by link, it needs a reset.

## Scorecard

Filed in `experiments/teacher_training/results/predictions_e41.json` before the run, unedited. Scored by the run's
`score` block and checked against the filed rules.

| id | claim | rule | read | verdict |
|---|---|---|---|---|
| C1 | unchanged model at its known floor | floor 0.0748 within 10% | 0.0748 | pass |
| G1, G3 | step-1,000 teachers and copies match E34 | P within 0.005; size within 0.001, ratio within 15% | equal to four digits | pass ×4 |
| C2 | chained start reproduces its start | first KL ≤ 0.005; weights within 1e-3 | 0.0005; 0 | pass |
| P1 | early task-text copies pass, gap ≤ 0.05 | every cell | emoticon step 10: 1.547, inconclusive; other three pass | **fails as filed** |
| P2 | early Pile-text copies learn, emoticon | learnt, between or neither | 0.998 and 0.995, both pass | learnt, as leaned |
| P3 | chain not worse, task text | 4 conditions per cell | 4 of 4 cells | holds |
| P4 | chain not worse, Pile text | 4 conditions per cell | 2 of 2 cells | holds |
| — | Δ untouched | Δ's share ≤ 1e-6 | ≤ 2.3e-9 | pass |

## Not run

- The step-10 emoticon copy at a second and third batch order; short chain links.
- A one-shot copy trained 3,000 steps to match the chain's total.
- SGD or LoRA teachers; other tasks; Pile-text copies on the " which" task.
- A second fine-tune seed; more adversary starts or a longer adversary.
- No states were saved; any follow-up regenerates one in about 3 minutes.

## Provenance

RunPod NVIDIA H100 80GB HBM3, 2026-10-28, job 12:15 to 12:58, 43.5 minutes of pod, about $3.2. Results
`experiments/teacher_training/results/teacher_training_e41.json`, committed in a450fde. Log
`experiments/teacher_training/results/e41_dl/run_e41.log`. Figures from `experiments/teacher_training/plot_e41.py`.